In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/all-MiniLM-L6-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 128 if device == "mps" else 64
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})

In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["label"] = df["label"].astype(np.float32)

print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())

In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()
print(model_name)

In [ ]:
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = df["label"].to_numpy(dtype=np.float32)

emb1 = model.encode(
    sentences1,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

emb2 = model.encode(
    sentences2,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)

In [ ]:
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic

results_df = df.copy()
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["error"] = results_df["predicted_score_0_5"] - results_df["label"]
results_df["absolute_error"] = np.abs(results_df["error"])

score_mean = float(np.mean(predicted_score_0_5))
score_std = float(np.std(predicted_score_0_5))
score_min = float(np.min(predicted_score_0_5))
score_median = float(np.median(predicted_score_0_5))
score_max = float(np.max(predicted_score_0_5))

label_mean = float(np.mean(labels))
label_std = float(np.std(labels))
label_min = float(np.min(labels))
label_median = float(np.median(labels))
label_max = float(np.max(labels))

absolute_error_values = results_df["absolute_error"].to_numpy(dtype=np.float32)
mae = float(np.mean(absolute_error_values))
rmse = float(np.sqrt(np.mean(np.square(results_df["error"].to_numpy(dtype=np.float32)))))

error_summary_df = pd.DataFrame({
    "metric": [
        "count", "mae", "rmse", "abs_error_std", "abs_error_min",
        "abs_error_p25", "abs_error_median", "abs_error_p75", "abs_error_p90",
        "abs_error_p95", "abs_error_max"
    ],
    "value": [
        int(len(results_df)),
        mae,
        rmse,
        float(np.std(absolute_error_values)),
        float(np.min(absolute_error_values)),
        float(np.percentile(absolute_error_values, 25)),
        float(np.median(absolute_error_values)),
        float(np.percentile(absolute_error_values, 75)),
        float(np.percentile(absolute_error_values, 90)),
        float(np.percentile(absolute_error_values, 95)),
        float(np.max(absolute_error_values)),
    ],
})

distribution_summary_df = pd.DataFrame({
    "series": ["predicted_score_0_5", "label"],
    "mean": [score_mean, label_mean],
    "std": [score_std, label_std],
    "min": [score_min, label_min],
    "median": [score_median, label_median],
    "max": [score_max, label_max],
})

print(results_df[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5", "error", "absolute_error"]].head(10))
print()
print("Score and label distribution summary:")
print(distribution_summary_df.to_string(index=False))
print()
print("Absolute-error summary:")
print(error_summary_df.to_string(index=False))

In [ ]:
top_k = 5

top_pred_pairs = results_df.nlargest(top_k, "predicted_score_0_5")[[
    "sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity", "error", "absolute_error"
]].reset_index(drop=True)

bottom_pred_pairs = results_df.nsmallest(top_k, "predicted_score_0_5")[[
    "sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity", "error", "absolute_error"
]].reset_index(drop=True)

worst_error_pairs = results_df.nlargest(top_k, "absolute_error")[[
    "sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity", "error", "absolute_error"
]].reset_index(drop=True)

print("Top prediction pairs:")
print(top_pred_pairs.to_string(index=False))
print()
print("Bottom prediction pairs:")
print(bottom_pred_pairs.to_string(index=False))
print()
print("Largest absolute-error pairs:")
print(worst_error_pairs.to_string(index=False))

In [ ]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"predicted_score_mean: {score_mean:.6f}")
print(f"predicted_score_std: {score_std:.6f}")
print(f"predicted_score_min: {score_min:.6f}")
print(f"predicted_score_median: {score_median:.6f}")
print(f"predicted_score_max: {score_max:.6f}")
print(f"label_mean: {label_mean:.6f}")
print(f"label_std: {label_std:.6f}")
print(f"label_min: {label_min:.6f}")
print(f"label_median: {label_median:.6f}")
print(f"label_max: {label_max:.6f}")
print(f"mae: {mae:.6f}")
print(f"rmse: {rmse:.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")